# Preprocesamiento de datos


In [ ]:
import re
import html
import string

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

for pkg, path in [
    ("stopwords", "corpora/stopwords"),
    ("punkt", "tokenizers/punkt"),
    ("punkt_tab", "tokenizers/punkt_tab"),
    ("wordnet", "corpora/wordnet"),
    ("omw-1.4", "corpora/omw-1.4"),
]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(pkg)

pd.set_option("display.max_colwidth", 200)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sofia\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sofia\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 1. Carga de los datos

Contiene 5 columnas:
`id`, `keyword`, `location`, `text` y `target` (1 = desastre real, 0 = no).

In [3]:
df = pd.read_csv("../data/train.csv")
print("Dimensiones:", df.shape)
df.head(10)

Dimensiones: (7613, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation orders in California",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school,1
5,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed in both directions due to Lake County fire - #CAfire #wildfires,1
6,10,NaN,NaN,"#flood #disaster Heavy rain causes flash flooding of streets in Manitou, Colorado Springs areas",1
7,13,NaN,NaN,I'm on top of the hill and I can see a fire in the woods...,1
8,14,NaN,NaN,There's an emergency evacuation happening now in the building across the street,1
9,15,NaN,NaN,I'm afraid that the tornado is coming to our area...,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


## 2. Revisión previa a la limpieza

In [5]:
faltantes = df.isna().sum().to_frame("n_faltantes")
faltantes["% del total"] = (faltantes["n_faltantes"] / len(df) * 100).round(1)
faltantes

,n_faltantes,% del total
id,0,0.0
keyword,61,0.8
location,2533,33.3
text,0,0.0
target,0,0.0


**Valores faltantes — decisión**

- `keyword`: 61 faltantes (0.8 %). Se rellenan con la cadena `"none"`.
- `location`: 2 533 faltantes (33 %). Es un campo escrito libremente por el usuario
  (mezcla de países, ciudades, bromas y texto sin sentido). No se usará como variable de
  texto para el modelo; se rellena con `"unknown"` y se conserva solo para el análisis exploratorio.
- `text` y `target`: sin faltantes. No se elimina ninguna fila por este motivo.

In [6]:
# 2 Duplicados exactos de texto
dup_text = df.duplicated(subset=["text"]).sum()
print("Filas con 'text' duplicado:", dup_text)

conflicto = df.groupby("text")["target"].nunique()
textos_conflictivos = conflicto[conflicto > 1].index
print("Textos con etiqueta contradictoria:", len(textos_conflictivos))
df[df["text"].isin(textos_conflictivos)].sort_values("text").head(10)

Filas con 'text' duplicado: 110
Textos con etiqueta contradictoria: 18


,id,keyword,location,text,target
4290,6094,hellfire,"Jubail IC, Saudi Arabia.",#Allah describes piling up #wealth thinking it would last #forever as the description of the people of #Hellfire in Surah Humaza. #Reflect,0
4299,6105,hellfire,?????? ??? ?????? ????????,#Allah describes piling up #wealth thinking it would last #forever as the description of the people of #Hellfire in Surah Humaza. #Reflect,0
4312,6123,hellfire,?????? ???? ??????,#Allah describes piling up #wealth thinking it would last #forever as the description of the people of #Hellfire in Surah Humaza. #Reflect,1
4244,6031,hazardous,"New Delhi, Delhi",#foodscare #offers2go #NestleIndia slips into loss after #Magginoodle #ban unsafe and hazardous for #humanconsumption,0
4221,5996,hazardous,NaN,#foodscare #offers2go #NestleIndia slips into loss after #Magginoodle #ban unsafe and hazardous for #humanconsumption,1
4239,6023,hazardous,"Mysore, Karnataka",#foodscare #offers2go #NestleIndia slips into loss after #Magginoodle #ban unsafe and hazardous for #humanconsumption,1
2832,4076,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #Genocide; refugees; IDP Internally displaced people; horror; etc. https://t.co/rqWuoy1fm4,0
2831,4072,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #Genocide; refugees; IDP Internally displaced people; horror; etc. https://t.co/rqWuoy1fm4,1
2830,4068,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #Genocide; refugees; IDP Internally displaced people; horror; etc. https://t.co/rqWuoy1fm4,1
2833,4077,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #Genocide; refugees; IDP Internally displaced people; horror; etc. https://t.co/rqWuoy1fm4,1


**Duplicados — decisión**

- **Etiquetas contradictorias (18 textos):** el mismo tweet aparece etiquetado como 0 y como 1.
  Como no hay forma de saber cuál es la correcta, se eliminan **todas** las filas con texto
  contradictorio para no meter ruido al entrenamiento.
- **Duplicados exactos (mismo texto y misma etiqueta):** se deja **una sola** copia. Repetir el
  mismo tweet infla artificialmente la frecuencia de esas palabras en los n-gramas.

In [7]:
n0 = len(df)

# Elimina textos con etiqueta contradictoria
df = df[~df["text"].isin(textos_conflictivos)].copy()

# Elimina duplicados exactos (texto + target), conservando la primera aparición
df = df.drop_duplicates(subset=["text", "target"], keep="first").copy()

df = df.reset_index(drop=True)
print(f"Filas eliminadas: {n0 - len(df)}   |   Filas restantes: {len(df)}")
print(df["target"].value_counts())

Filas eliminadas: 128   |   Filas restantes: 7485
target
0    4297
1    3188
Name: count, dtype: int64


## 3. Limpieza y preprocesamiento del texto

Pasos de la limpieza del texto crudo:

1. **Des-escape de HTML.** Los tweets traen `&amp;`, `&gt;`, `&lt;` → se convierten a `&`, `>`, `<`.
2. **Minúsculas.** `Fire`, `FIRE` y `fire` deben contar como la misma palabra.
3. **Eliminar URLs.** Patrones `http://…`, `https://…`, `www.…`. Un enlace acortado no aporta
   información léxica para clasificar.
4. **Eliminar menciones `@usuario`.** Identifican a una cuenta, no al contenido del tweet.
5. **Hashtags:** se elimina solo el símbolo `#` y se conserva la palabra
   (`#earthquake` → `earthquake`). El texto del hashtag sí es informativo para detectar desastres.
6. **Eliminar emoticones de texto** (`:)`, `:-(`, `;)`, `:D`, `<3`, …) y **emojis Unicode**.
   Para esta etapa (n-gramas y modelo preliminar) se quitan; en la parte de análisis de
   sentimiento se evaluará por separado si conviene conservarlos.
7. **Expandir contracciones frecuentes** (`i'm`→`i am`, `don't`→`do not`, `can't`→`can not`…)
   y luego **eliminar los apóstrofes** restantes.
8. **Números:** se eliminan todos **excepto `911`**. La mayoría de números (fechas, conteos,
   códigos) son ruido, pero `911` es el número de emergencias en EE. UU. y aparece en tweets de
   desastres reales, así que se conserva como token `911`.
9. **Eliminar puntuación y caracteres especiales** (`#`, `@`, `"`, `-`, `…`, etc.): se deja solo
   letras, dígitos y espacios.
10. **Normalizar espacios en blanco** (saltos de línea, tabuladores y espacios múltiples → un espacio).

In [8]:
URL_RE      = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_RE  = re.compile(r"@\w+")
HASHTAG_RE  = re.compile(r"#(\w+)")

EMOTICON_RE = re.compile(r"[:;=8][\-o\*']?[\)\]\(\[dDpP\\\|]+|<3|\^_\^|-_-|:'\(")

EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002190-\U000021FF"
    "\U00002B00-\U00002BFF"
    "\U0000FE0F"
    "]+",
    flags=re.UNICODE,
)
NUMBER_RE   = re.compile(r"\b\d+\b")
NON_ALNUM_RE = re.compile(r"[^a-z0-9\s]")
SPACES_RE   = re.compile(r"\s+")


CONTRACCIONES = [
    (re.compile(r"\bwon't\b"), "will not"),
    (re.compile(r"\bcan't\b"), "can not"),
    (re.compile(r"\bi'm\b"), "i am"),
    (re.compile(r"\blet's\b"), "let us"),
    (re.compile(r"n't\b"), " not"),
    (re.compile(r"'re\b"), " are"),
    (re.compile(r"'ll\b"), " will"),
    (re.compile(r"'ve\b"), " have"),
    (re.compile(r"'d\b"), " would"),
]

def limpiar_texto(texto: str) -> str:
    t = str(texto)
    t = html.unescape(t)                 
    t = t.lower()                      
    t = URL_RE.sub(" ", t)               
    t = MENTION_RE.sub(" ", t)        
    t = HASHTAG_RE.sub(r"\1", t)       
    t = EMOJI_RE.sub(" ", t)      
    for pat, repl in CONTRACCIONES:     
        t = pat.sub(repl, t)
    t = t.replace("'", "")             
    t = re.sub(r"\b(?!911\b)\d+\b", " ", t)      
    t = NON_ALNUM_RE.sub(" ", t)     
    t = SPACES_RE.sub(" ", t).strip()  
    return t

### 3.1 Demostración paso a paso

Se aplica la función a algunos tweets representativos (con hashtags, URL, mención, emoticón
y el caso del `911`) para verificar el resultado.

In [9]:
ejemplos = [
    "Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all",
    "@bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C",
    "13,000 people receive #wildfires evacuation orders in California ",
    "London is cool ;)",
    "Call 911 now!! There's a fire &amp; smoke everywhere http://t.co/xyz",
]
pd.DataFrame({"original": ejemplos, "limpio": [limpiar_texto(e) for e in ejemplos]})

,original,limpio
0,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,our deeds are the reason of this earthquake may allah forgive us all
1,@bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C,wholesale markets ablaze
2,"13,000 people receive #wildfires evacuation orders in California",people receive wildfires evacuation orders in california
3,London is cool ;),london is cool
4,Call 911 now!! There's a fire &amp; smoke everywhere http://t.co/xyz,call 911 now theres a fire smoke everywhere


### 3.2 Aplicar la limpieza a todo el dataset

In [10]:
df["text_clean"] = df["text"].apply(limpiar_texto)
df[["text", "text_clean", "target"]].head(10)

,text,text_clean,target
0,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,our deeds are the reason of this earthquake may allah forgive us all,1
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada,1
2,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected,all residents asked to shelter in place are being notified by officers no other evacuation or shelter in place orders are expected,1
3,"13,000 people receive #wildfires evacuation orders in California",people receive wildfires evacuation orders in california,1
4,Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school,just got sent this photo from ruby alaska as smoke from wildfires pours into a school,1
5,#RockyFire Update => California Hwy. 20 closed in both directions due to Lake County fire - #CAfire #wildfires,rockyfire update california hwy closed in both directions due to lake county fire cafire wildfires,1
6,"#flood #disaster Heavy rain causes flash flooding of streets in Manitou, Colorado Springs areas",flood disaster heavy rain causes flash flooding of streets in manitou colorado springs areas,1
7,I'm on top of the hill and I can see a fire in the woods...,i am on top of the hill and i can see a fire in the woods,1
8,There's an emergency evacuation happening now in the building across the street,theres an emergency evacuation happening now in the building across the street,1
9,I'm afraid that the tornado is coming to our area...,i am afraid that the tornado is coming to our area,1


### 3.3 Limpieza de la columna `keyword`

La palabra clave viene con `%20` en lugar de espacios (p. ej. `airplane%20accident`).
Se decodifica y se pasa a minúsculas. Los faltantes se rellenan con `"none"`.

In [11]:
df["keyword"] = (
    df["keyword"].fillna("none")
    .str.replace("%20", " ", regex=False)
    .str.lower()
    .str.strip()
)
df["location"] = df["location"].fillna("unknown").str.strip()
df["keyword"].value_counts().head(10)

keyword
none          56
fatalities    45
deluge        42
armageddon    42
damage        41
body bags     41
harm          41
siren         40
evacuate      40
twister       40
Name: count, dtype: int64

## 4. Tokenización

Sobre el texto ya limpio:

1. **Tokenización** con `nltk.word_tokenize`: separa la cadena en palabras individuales.
2. **Eliminar stopwords:** artículos, preposiciones, conjunciones y pronombres en inglés
   (`the`, `a`, `to`, `and`, `of`, `in`…) que aparecen en todos los tweets y no ayudan a
   distinguir categorías. Se usa la lista de `nltk.corpus.stopwords`.
   - Se **conservan** algunas palabras de negación (`no`, `not`, `nor`) porque sí cambian el
     sentido y serán útiles para el análisis de sentimiento.
3. **Eliminar tokens de 1–2 letras** que quedan como residuo tras la limpieza.
4. **Lematización** con `WordNetLemmatizer`: lleva cada palabra a su forma base
   (`fires`, `burning` → `fire`, `burn`). Se prefiere lematización sobre *stemming* porque
   produce palabras reales, más legibles para el análisis exploratorio y las nubes de palabras.

In [12]:
STOP_EN = set(stopwords.words("english"))

STOP_EN -= {"no", "not", "nor"}

lem = WordNetLemmatizer()

def tokenizar(texto: str) -> list:
    tokens = word_tokenize(texto)
    tokens = [tok for tok in tokens if tok not in STOP_EN and len(tok) > 2 or tok == "911"]
    tokens = [lem.lemmatize(tok) for tok in tokens]
    return tokens

df["tokens"] = df["text_clean"].apply(tokenizar)
df["text_final"] = df["tokens"].apply(lambda toks: " ".join(toks))
df[["text_clean", "text_final", "target"]].head(10)

,text_clean,text_final,target
0,our deeds are the reason of this earthquake may allah forgive us all,deed reason earthquake may allah forgive,1
1,forest fire near la ronge sask canada,forest fire near ronge sask canada,1
2,all residents asked to shelter in place are being notified by officers no other evacuation or shelter in place orders are expected,resident asked shelter place notified officer evacuation shelter place order expected,1
3,people receive wildfires evacuation orders in california,people receive wildfire evacuation order california,1
4,just got sent this photo from ruby alaska as smoke from wildfires pours into a school,got sent photo ruby alaska smoke wildfire pours school,1
5,rockyfire update california hwy closed in both directions due to lake county fire cafire wildfires,rockyfire update california hwy closed direction due lake county fire cafire wildfire,1
6,flood disaster heavy rain causes flash flooding of streets in manitou colorado springs areas,flood disaster heavy rain cause flash flooding street manitou colorado spring area,1
7,i am on top of the hill and i can see a fire in the woods,top hill see fire wood,1
8,theres an emergency evacuation happening now in the building across the street,there emergency evacuation happening building across street,1
9,i am afraid that the tornado is coming to our area,afraid tornado coming area,1


## 5. Revisión posterior a la limpieza

### 5.1 Tweets que quedaron vacíos

Algunos tweets estaban formados casi solo por URLs, menciones o emojis; tras la limpieza
quedan sin texto. Se revisan y se eliminan (no aportan nada a los n-gramas ni al modelo).

In [13]:
vacios = df[df["text_final"].str.len() == 0]
print("Tweets vacíos tras la limpieza:", len(vacios))
display(vacios[["text", "target"]].head(10))

df = df[df["text_final"].str.len() > 0].reset_index(drop=True)
print("Filas finales:", len(df))

Tweets vacíos tras la limpieza: 1


,text,target
3603,@Truly_Stings Yo Dm me,1


Filas finales: 7484


In [14]:
df["n_palabras_orig"]  = df["text"].str.split().apply(len)
df["n_palabras_final"] = df["tokens"].apply(len)
df[["n_palabras_orig", "n_palabras_final"]].describe().round(2)

,n_palabras_orig,n_palabras_final
count,7484.00,7484.00
mean,14.87,8.51
std,5.73,3.44
min,1.00,1.00
25%,11.00,6.00
50%,15.00,9.00
75%,19.00,11.00
max,31.00,21.00


## 6. Guardar el dataset preprocesado

Se guarda `train_clean.csv` con las columnas originales más:

- `text_clean`: texto limpio (sin URLs, hashtags, puntuación, números salvo 911) pero **con** stopwords.
- `text_final`: texto limpio + sin stopwords + lematizado. **Esta es la que usarán las etapas de
  n-gramas y modelado.**
- `tokens`: la versión en lista de `text_final`.
- `n_palabras_orig`, `n_palabras_final`: longitudes para el análisis exploratorio.

In [15]:
cols = ["id", "keyword", "location", "text",
        "text_clean", "text_final", "tokens",
        "n_palabras_orig", "n_palabras_final", "target"]
df_out = df[cols].copy()
df_out["tokens"] = df_out["tokens"].apply(lambda t: " ".join(t))  
df_out.to_csv("../data/train_clean.csv", index=False)
print("Guardado train_clean.csv con", df_out.shape[0], "filas y", df_out.shape[1], "columnas")
df_out.head()

Guardado train_clean.csv con 7484 filas y 10 columnas


,id,keyword,location,text,text_clean,text_final,tokens,n_palabras_orig,n_palabras_final,target
0,1,none,unknown,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,our deeds are the reason of this earthquake may allah forgive us all,deed reason earthquake may allah forgive,deed reason earthquake may allah forgive,13,6,1
1,4,none,unknown,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada,forest fire near ronge sask canada,forest fire near ronge sask canada,7,6,1
2,5,none,unknown,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected,all residents asked to shelter in place are being notified by officers no other evacuation or shelter in place orders are expected,resident asked shelter place notified officer evacuation shelter place order expected,resident asked shelter place notified officer evacuation shelter place order expected,22,11,1
3,6,none,unknown,"13,000 people receive #wildfires evacuation orders in California",people receive wildfires evacuation orders in california,people receive wildfire evacuation order california,people receive wildfire evacuation order california,8,6,1
4,7,none,unknown,Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school,just got sent this photo from ruby alaska as smoke from wildfires pours into a school,got sent photo ruby alaska smoke wildfire pours school,got sent photo ruby alaska smoke wildfire pours school,16,9,1


## 7. Resumen del preprocesamiento

| # | Actividad | Herramienta | Decisión tomada |
|---|---|---|---|
| 1 | Carga de `train.csv` | `pandas` | 7 613 filas, 5 columnas |
| 2 | Valores faltantes | `pandas` | `keyword`→`"none"`, `location`→`"unknown"`; sin faltantes en `text`/`target` |
| 3 | Etiquetas contradictorias | `pandas` | se eliminan los 18 textos con `target` 0 y 1 a la vez |
| 4 | Duplicados exactos | `pandas` | se conserva una sola copia de cada (texto, target) |
| 5 | Des-escape HTML | `html` | `&amp;`→`&`, etc. |
| 6 | Minúsculas | `str.lower` | unifica mayúsculas/minúsculas |
| 7 | URLs | `re` | eliminadas (≈ 3 900 tweets las tenían) |
| 8 | Menciones `@usuario` | `re` | eliminadas |
| 9 | Hashtags | `re` | se quita `#`, se conserva la palabra |
| 10 | Emoticones y emojis | `re` | eliminados en esta etapa (se re-evalúan en sentimiento) |
| 11 | Contracciones y apóstrofes | `re` | `i'm`→`i am`, …; apóstrofes eliminados |
| 12 | Números | `re` | eliminados **salvo `911`** (número de emergencias) |
| 13 | Puntuación / caracteres especiales | `re`, `string` | se deja solo letras, dígitos y espacios |
| 14 | Espacios en blanco | `re` | normalizados a un espacio |
| 15 | Tokenización | `nltk.word_tokenize` | palabras individuales |
| 16 | Stopwords | `nltk.corpus.stopwords` | eliminadas (se conservan `no`, `not`, `nor`) |
| 17 | Tokens muy cortos | — | eliminados los de ≤ 2 letras |
| 18 | Lematización | `nltk.WordNetLemmatizer` | forma base de cada palabra (preferida sobre stemming) |
| 19 | Tweets vacíos tras limpieza | `pandas` | eliminados |
| 20 | Exportación | `pandas` | `train_clean.csv` |

**Salida:** `train_clean.csv`, con `text_final` como columna de texto lista para generar
unigramas / bigramas y entrenar los modelos de clasificación.